# Data Labeler: Determining Article Quality
This notebook creates an algorithm to determine if a financial article is "Good" or "Rubbish" (Bullish vs Bearish) based on the subsequent movement of the mentioned stock's price.
It cross-references the dates in `perfectly_balanced_stock_news.csv` with the historical stock prices found in the `csv-history` folder.


In [1]:
import pandas as pd
import numpy as np
import os
from datetime import timedelta


In [2]:
# 1. Load the news dataset
news_df = pd.read_csv('../csv-history/perfectly_balanced_stock_news.csv')

# Parse the dates. Example format: 2023-12-16 22:00:00 UTC
news_df['Parsed_Date'] = pd.to_datetime(news_df['Date'], format='mixed', utc=True).dt.tz_localize(None)
news_df['Just_Date'] = news_df['Parsed_Date'].dt.floor('D')

print(f"Loaded {len(news_df)} articles.")
news_df.head(2)


Loaded 4726 articles.


,Unnamed: 0,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary,Parsed_Date,Just_Date
0,12025.0,2023-12-16 22:00:00 UTC,My 6 Largest Portfolio Holdings Heading Into 2...,AAPL,https://www.nasdaq.com/articles/my-6-largest-p...,NaN,NaN,"After an absolute disaster of a year in 2022, ...",3: Apple There's little question that Apple (N...,3: Apple There's little question that Apple (N...,3: Apple There's little question that Apple (N...,3: Apple There's little question that Apple (N...,2023-12-16 22:00:00,2023-12-16
1,12026.0,2023-12-16 22:00:00 UTC,Brokers Suggest Investing in Apple (AAPL): Rea...,AAPL,https://www.nasdaq.com/articles/brokers-sugges...,NaN,NaN,"When deciding whether to buy, sell, or hold a ...",Let's take a look at what these Wall Street he...,Click to get this free report Apple Inc. (AAPL...,Let's take a look at what these Wall Street he...,Brokerage Recommendation Trends for AAPL Let's...,2023-12-16 22:00:00,2023-12-16


In [5]:
# 2. Pre-load all stock histories into a dictionary
stock_files = [f for f in os.listdir('../csv-history') if f.endswith('.csv') and f != 'perfectly_balanced_stock_news.csv' and f != 'perfectly_balanced_stock_news_labeled.csv']
stock_data = {}

for file in stock_files:
    symbol = file.replace('.csv', '').upper()
    df = pd.read_csv(f"../csv-history/{file}")
    df['date'] = pd.to_datetime(df['date'])
    # Sort by date ascending so we can easily look ahead
    df = df.sort_values('date').reset_index(drop=True)
    stock_data[symbol] = df

print(f"Loaded history for symbols: {list(stock_data.keys())}")


Loaded history for symbols: ['AAPL', 'AMZN', 'BRKL', 'GOOGL', 'JPM', 'LLY', 'MSFT', 'NVDA', 'TSLA', 'WMT']


In [6]:
# 3. Algorithm to determine if an article is "Good" or "Rubbish"
# Logic: We'll compare the closing price on the next available trading day after publication
# to the closing price 'N' trading days later. If the stock goes up, it's "Good" (Bullish), else "Rubbish" (Bearish).

LOOKAHEAD_DAYS = 3 # Look 3 trading days ahead

def evaluate_article(row):
    symbol = str(row['Stock_symbol']).upper()
    pub_date = row['Just_Date']
    
    if symbol not in stock_data:
        return 'Unknown'
        
    history = stock_data[symbol]
    
    # Find all trading days >= the publication date
    future_dates = history[history['date'] >= pub_date]
    
    # We need at least (LOOKAHEAD_DAYS + 1) days of data to compare
    if len(future_dates) < LOOKAHEAD_DAYS + 1:
        return 'Unknown'
        
    # Price on publication day (or nearest next trading day)
    initial_price = future_dates.iloc[0]['close']
    
    # Price N trading days later
    future_price = future_dates.iloc[LOOKAHEAD_DAYS]['close']
    
    # Calculate percentage change
    pct_change = (future_price - initial_price) / initial_price
    
    # Return Good if it goes up, Rubbish if it drops
    if pct_change > 0:
        return 'Good'
    else:
        return 'Rubbish'

# Apply the algorithm
print("Evaluating articles... This may take a minute.")
news_df['Article_Quality'] = news_df.apply(evaluate_article, axis=1)

print("Evaluation complete!")
print(news_df['Article_Quality'].value_counts())


Evaluating articles... This may take a minute.
Evaluation complete!
Article_Quality
Good       2542
Rubbish    1970
Unknown     214
Name: count, dtype: int64


In [8]:
# 4. Save the new labeled dataset
# We can drop the temporary date columns and save to a new CSV
final_df = news_df.drop(columns=['Parsed_Date', 'Just_Date'])

# Filter out 'Unknown' if you only want labeled rows
labeled_df = final_df[final_df['Article_Quality'] != 'Unknown']

output_path = '../csv-history/perfectly_balanced_stock_news_labeled.csv'
labeled_df.to_csv(output_path, index=False)

print(f"Labeled dataset saved to {output_path} with {len(labeled_df)} rows.")


Labeled dataset saved to ../csv-history/perfectly_balanced_stock_news_labeled.csv with 4512 rows.
